# Tiền xử lý dữ liệu cho mô hình GAT-GRU

**Đề tài:** Tối ưu hóa autoscaling chủ động cho microservices bằng GNN và học tăng cường.

Trong vòng lặp MAPE-K của đề tài, notebook này thuộc pha **Monitor → Analyze**. Nó biến dữ liệu thô mà `run-scenario.sh` thu được (Prometheus + Jaeger + Locust) thành tập dữ liệu đồ thị theo thời gian cho GAT-GRU:

$$\hat{y}_{t+1:t+h} = f_{\text{GAT-GRU}}\left(\mathcal{G}_{t-w+1:t}\right), \qquad \mathcal{G}_\tau = (V, E, X_\tau, E^{attr}_\tau)$$

- **Nút $V$**: 11 microservice của Online Boutique.
- **Cạnh $E$**: quan hệ gọi dịch vụ, gồm topology tĩnh và các cạnh quan sát được từ trace Jaeger/Istio.
- **$X_\tau$**: đặc trưng của từng nút tại bước $\tau$ (10s một bước).
- **$E^{attr}_\tau$**: đặc trưng động của cạnh, gồm tần suất gọi, độ trễ và lỗi. Các đặc trưng này làm tín hiệu để lớp GAT học trọng số phụ thuộc.
- **$\hat y$**: nhu cầu tải (`rps_in`) và nhu cầu tài nguyên (`cpu_cores`) của từng service tại $h$ bước tới. Đánh giá bằng MAE/RMSE trên đơn vị gốc.

Đầu ra của dự báo sẽ được đưa vào **state của tác tử PPO** (metric hiện tại, nhu cầu dự báo, replica, trạng thái SLO). Vì vậy notebook còn xuất bảng dữ liệu raw đã căn chỉnh để xây môi trường RL offline.

**Quy trình:** nạp dữ liệu → kiểm tra chất lượng → căn lưới thời gian → tạo đặc trưng → xây Service Graph → làm sạch → chia tập theo run → chuẩn hóa → cắt cửa sổ → lưu.

In [ ]:
import json
import math
import tempfile
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 40)
pd.set_option('display.width', 160)
plt.rcParams.update({'figure.dpi': 110, 'axes.grid': True, 'grid.alpha': 0.3})

## 1. Cấu hình

| Tham số | Giá trị | Lý do |
|---|---|---|
| `STEP_SEC` | 10s | Bằng `scrape_interval` của Prometheus và bước xuất của `collect_metrics.py` |
| `WINDOW` (w) | 12 bước = 2 phút | Đủ dài để GRU thấy xu hướng tăng/giảm và trọn một cửa sổ burst 30s. Cũng đủ ngắn để controller phản ứng nhanh |
| `HORIZON` (h) | 6 bước = 60s | Một pod mới cần khoảng 30–60s (schedule, khởi động, sidecar, readiness 60s với Python service). Muốn scale **chủ động** thì phải biết trước ít nhất khoảng thời gian này |
| `WARMUP_DROP` | 6 bước | `rate()[40s]` chưa đủ mẫu và Locust đang spawn user nên dữ liệu đầu run bị méo |
| `SPLIT_MODE` | `by_run` | Chia theo **run**, không cắt ngẫu nhiên từng cửa sổ. Các cửa sổ chồng lấn trong cùng một run sẽ làm rò rỉ dữ liệu giữa train và test |

In [ ]:
RESULTS_DIR = Path('results')        # load-testing/results/<scenario>/run_XX/
OUTPUT_DIR = Path('processed')       # dataset sau tiền xử lý
# Chưa có run thật thì tự chuyển sang dữ liệu giả lập để chạy thử toàn bộ notebook.
DEMO = not any(RESULTS_DIR.glob('*/run_*/meta.json'))

STEP_SEC = 10
WINDOW = 12
HORIZON = 6
WARMUP_DROP = 6
LATENCY_SLO_MS = 300.0
ERROR_RATE_SLO = 0.01
JAEGER_COVERAGE_MIN = 0.9   # dưới ngưỡng này: Jaeger mất span, rps/call_rate bị đếm thiếu
CORRECT_COVERAGE = False    # True: nhân rps_in/call_rate với 1/coverage của run

SPLIT_MODE = 'by_run'       # 'by_run' | 'by_scenario'
SPLIT_FRAC = {'train': 0.70, 'val': 0.15, 'test': 0.15}
TEST_SCENARIOS = ['spike']  # chỉ dùng khi SPLIT_MODE = 'by_scenario' (đánh giá ngoài phân phối)
TARGETS = ['rps_in', 'cpu_cores']
SEED = 42

SERVICES = [
    'frontend', 'adservice', 'cartservice', 'checkoutservice', 'currencyservice',
    'emailservice', 'paymentservice', 'productcatalogservice', 'recommendationservice',
    'redis-cart', 'shippingservice',
]
# Topology Online Boutique (GoogleCloudPlatform/microservices-demo)
STATIC_EDGES = [
    ('frontend', 'productcatalogservice'), ('frontend', 'currencyservice'),
    ('frontend', 'cartservice'), ('frontend', 'recommendationservice'),
    ('frontend', 'shippingservice'), ('frontend', 'checkoutservice'),
    ('frontend', 'adservice'), ('recommendationservice', 'productcatalogservice'),
    ('checkoutservice', 'paymentservice'), ('checkoutservice', 'productcatalogservice'),
    ('checkoutservice', 'shippingservice'), ('checkoutservice', 'emailservice'),
    ('checkoutservice', 'currencyservice'), ('checkoutservice', 'cartservice'),
    ('cartservice', 'redis-cart'),
]
SVC_IDX = {s: i for i, s in enumerate(SERVICES)}
print('DEMO mode:', DEMO)

## 2. Dữ liệu giả lập (chỉ chạy khi `DEMO = True`)

Khi chưa có dữ liệu thật, ô dưới sinh các run **đúng định dạng file** mà `run-scenario.sh` / `collect_metrics.py` tạo ra (`meta.json`, `node_metrics.csv`, `service_metrics.csv`, `edge_metrics.csv`, `locust_stats_history.csv`). Nhờ đó cả notebook có thể chạy thử từ đầu đến cuối.

Mô phỏng gồm: lượng user theo 3 kịch bản → RPS frontend → lan truyền xuống các service theo tỉ lệ gọi → CPU, throttle, latency tăng khi gần bão hòa → một HPA đơn giản thay đổi replica. **Số liệu demo không có giá trị khoa học.**

In [ ]:
def make_demo_results(root: Path, runs_per_scenario=4, duration_sec=1800, seed=SEED):
    rng = np.random.default_rng(seed)
    ratio = {  # số lời gọi downstream trên mỗi request upstream (ước lượng)
        ('frontend', 'productcatalogservice'): 1.6, ('frontend', 'currencyservice'): 1.9,
        ('frontend', 'cartservice'): 0.7, ('frontend', 'recommendationservice'): 0.45,
        ('frontend', 'shippingservice'): 0.15, ('frontend', 'checkoutservice'): 0.04,
        ('frontend', 'adservice'): 0.45, ('recommendationservice', 'productcatalogservice'): 1.0,
        ('checkoutservice', 'paymentservice'): 1.0, ('checkoutservice', 'productcatalogservice'): 1.5,
        ('checkoutservice', 'shippingservice'): 2.0, ('checkoutservice', 'emailservice'): 1.0,
        ('checkoutservice', 'currencyservice'): 2.5, ('checkoutservice', 'cartservice'): 2.0,
    }
    cpu_ms = {'frontend': 6, 'adservice': 4, 'cartservice': 3, 'checkoutservice': 5, 'currencyservice': 2,
              'emailservice': 6, 'paymentservice': 2, 'productcatalogservice': 1.5,
              'recommendationservice': 8, 'redis-cart': 0.5, 'shippingservice': 1.5}
    limit = {s: (1.0 if s in ('emailservice', 'recommendationservice') else 0.5) for s in SERVICES}
    request = {s: (0.1 if s == 'emailservice' else 0.15) for s in SERVICES}
    order = ['frontend', 'recommendationservice', 'checkoutservice']
    T = duration_sec // STEP_SEC
    t0 = 1_790_000_000

    def users_curve(sc, r):
        t = np.arange(T) * STEP_SEC
        if sc == 'normal':
            return np.where(t < 60, 40 * t / 60, 40 * (1 + r.uniform(-0.1, 0.1, T)))
        if sc == 'spike':
            return np.where(t % 360 < 90, 220.0, 25.0)
        draws, u = {}, np.empty(T)
        for i, w in enumerate(t // 30):
            if w not in draws:
                draws[w] = r.integers(80, 181) if r.random() < 0.35 else 20 * r.uniform(0.7, 1.3)
            u[i] = draws[w]
        return u

    for sc in ['normal', 'spike', 'bursty']:
        for k in range(1, runs_per_scenario + 1):
            d = root / sc / f'run_{k:02d}'
            d.mkdir(parents=True, exist_ok=True)
            start = t0
            t0 += duration_sec + 300
            users = users_curve(sc, rng)
            rps = {s: np.zeros(T) for s in SERVICES}
            rps['frontend'] = users / 2.0 * rng.uniform(0.95, 1.05, T)
            edge_rps = {}
            for src in order:
                for (a, b), q in ratio.items():
                    if a == src:
                        edge_rps[(a, b)] = rps[a] * q * rng.uniform(0.9, 1.1, T)
                        rps[b] = rps[b] + edge_rps[(a, b)]
            replicas = {s: np.ones(T) for s in SERVICES}
            node_rows, svc_rows, edge_rows = [], [], []
            restarts = {s: 0 for s in SERVICES}
            rps['redis-cart'] = rps['cartservice'] * 1.2
            util = {s: np.zeros(T) for s in SERVICES}
            for i in range(T):
                ts = start + i * STEP_SEC
                for s in SERVICES:
                    rep = replicas[s][i]
                    demand = 0.01 + rps[s][i] * cpu_ms[s] / 1000 * rng.uniform(0.9, 1.1)
                    cap = limit[s] * rep
                    cpu = min(demand, cap * 0.98)
                    util[s][i] = cpu / (request[s] * rep)
                    thr = float(np.clip((demand - cap) / max(demand, 1e-9) + 0.02 * rng.random(), 0, 1))
                    sat = demand / cap
                    p50 = (2 + 3 * cpu_ms[s]) * (1 + 4 * max(0, sat - 0.7) ** 2)
                    p95 = p50 * 2.5 * (1 + 6 * max(0, sat - 0.8) ** 2)
                    err = rps[s][i] * STEP_SEC * (0.001 + 0.2 * max(0, sat - 1.0))
                    if sat > 1.3 and rng.random() < 0.01:
                        restarts[s] += 1
                    node_rows.append([ts, s, cpu, 0.002 + rps[s][i] * 0.0004, thr,
                                      (60 + 30 * rep + 0.5 * rps[s][i]) * 2**20,
                                      rps[s][i] * 1800, rps[s][i] * 2600, request[s] * rep, limit[s] * rep,
                                      restarts[s], rep, rep])
                    if s != 'redis-cart' and rps[s][i] > 0:
                        n = int(rng.poisson(rps[s][i] * STEP_SEC))
                        svc_rows.append([ts, s, n, int(min(n, rng.poisson(err))), p50, p95])
                    # HPA đơn giản: mục tiêu 70% CPU request, quyết định mỗi 30s, trễ 1 bước
                    if i + 1 < T:
                        replicas[s][i + 1] = rep
                        if i % 3 == 0 and s != 'redis-cart':
                            want = int(np.clip(np.ceil(util[s][i] * rep / 0.7 / 3.0), 1, 4))
                            replicas[s][i + 1] = want
                for (a, b), arr in edge_rps.items():
                    if b == 'redis-cart':
                        continue  # TCP: không có span
                    n = int(rng.poisson(arr[i] * STEP_SEC))
                    if n:
                        lat = (2 + 3 * cpu_ms[b]) * 2.8 + rng.gamma(2, 2)
                        edge_rows.append([ts, a, b, n, int(rng.poisson(n * 0.002)), lat * 0.45, lat])
            pd.DataFrame(node_rows, columns=['timestamp', 'service', 'cpu_cores', 'cpu_sidecar_cores',
                                             'cpu_throttle_ratio', 'mem_bytes', 'net_rx_bps', 'net_tx_bps',
                                             'cpu_request_cores', 'cpu_limit_cores', 'restarts_total',
                                             'replicas', 'replicas_desired']).to_csv(d / 'node_metrics.csv', index=False)
            pd.DataFrame(svc_rows, columns=['timestamp', 'service', 'request_count', 'error_count',
                                            'latency_p50_ms', 'latency_p95_ms']).to_csv(d / 'service_metrics.csv', index=False)
            pd.DataFrame(edge_rows, columns=['timestamp', 'source', 'target', 'call_count', 'error_count',
                                             'latency_p50_ms', 'latency_p95_ms']).to_csv(d / 'edge_metrics.csv', index=False)
            sec = np.arange(duration_sec)
            u_sec = np.repeat(users, STEP_SEC)[:duration_sec]
            pd.DataFrame({'Timestamp': start + sec, 'User Count': u_sec.astype(int), 'Type': '',
                          'Name': 'Aggregated', 'Requests/s': u_sec / 2.0, 'Failures/s': 0.0,
                          '95%': 120.0}).to_csv(d / 'locust_stats_history.csv', index=False)
            json.dump({'scenario': sc, 'run_id': d.name, 'start': start, 'end': start + duration_sec,
                       'duration_sec': duration_sec, 'step_sec': STEP_SEC, 'locust_exit_code': 0},
                      open(d / 'meta.json', 'w'))
    return root


if DEMO:
    RESULTS_DIR = make_demo_results(Path(tempfile.mkdtemp(prefix='gatgru_demo_')))
    OUTPUT_DIR = RESULTS_DIR.parent / (RESULTS_DIR.name + '_processed')
print('RESULTS_DIR =', RESULTS_DIR)

## 3. Nạp dữ liệu thô

Mỗi run nằm ở `results/<scenario>/run_XX/`:

| File | Nguồn | Nội dung |
|---|---|---|
| `meta.json` | `run-scenario.sh` | kịch bản, thời điểm bắt đầu/kết thúc (unix time) |
| `node_metrics.csv` | Prometheus: cAdvisor (3 node) + kube-state-metrics | CPU, memory, network, throttle, request/limit, replicas, restarts theo service |
| `service_metrics.csv` | Jaeger: span `server` của Envoy | số request vào, lỗi, p50/p95 latency theo service |
| `edge_metrics.csv` | Jaeger: span `client` của Envoy | số lời gọi, lỗi, latency theo cặp `source → target` |
| `locust_stats_history.csv` | Locust | user count, RPS và p95 phía client. **Chỉ dùng để kiểm tra**, không đưa vào mô hình |

In [ ]:
def _read(path, **kw):
    return pd.read_csv(path, **kw) if path.exists() and path.stat().st_size > 0 else pd.DataFrame()


def load_runs(root: Path):
    runs = []
    for meta_path in sorted(root.glob('*/run_*/meta.json')):
        d = meta_path.parent
        meta = json.loads(meta_path.read_text())
        loc = _read(d / 'locust_stats_history.csv')
        if not loc.empty and 'Name' in loc:
            loc = loc[loc['Name'] == 'Aggregated']
        runs.append({
            'scenario': meta['scenario'], 'run_id': meta.get('run_id', d.name), 'dir': d, 'meta': meta,
            'node': _read(d / 'node_metrics.csv'), 'svc': _read(d / 'service_metrics.csv'),
            'edge': _read(d / 'edge_metrics.csv'), 'locust': loc,
            'report': json.loads((d / 'collect_report.json').read_text()) if (d / 'collect_report.json').exists() else {},
        })
    return runs


runs = load_runs(RESULTS_DIR)
assert runs, f'Không tìm thấy run nào trong {RESULTS_DIR}'
pd.DataFrame([{'scenario': r['scenario'], 'run': r['run_id'],
               'duration_min': (r['meta']['end'] - r['meta']['start']) / 60,
               'node_rows': len(r['node']), 'service_rows': len(r['svc']), 'edge_rows': len(r['edge']),
               'locust_rows': len(r['locust'])} for r in runs])

## 4. Căn lưới thời gian và kiểm tra chất lượng

Mỗi run được căn về lưới $t = 0, 1, \dots, T-1$ (bước 10s), với mốc là `start` làm tròn xuống bội số của `STEP_SEC`. Sau đó kiểm tra:

1. **Độ phủ Prometheus**: tỉ lệ ô (bước × service) có dữ liệu CPU. Thiếu nhiều nghĩa là target kubelet bị down hoặc pod đang restart.
2. **Độ phủ Jaeger**: so tổng request vào `frontend` đếm được từ span với tổng request Locust gửi. Jaeger chạy trên VM 1 vCPU nên có thể **rơi span** khi tải cao. Khi đó `rps_in`/`call_rate` bị đếm thiếu một cách có hệ thống.
3. **Cửa sổ trace bị cắt** (`jaeger_truncated_windows` trong `collect_report.json`): số trace vượt `TRACE_LIMIT` ngay cả sau khi đã chia nhỏ cửa sổ.

In [ ]:
def to_grid(df, meta, col='timestamp'):
    start = meta['start'] - meta['start'] % STEP_SEC
    out = df.copy()
    out['t'] = ((out[col] - start) // STEP_SEC).astype(int)
    return out


quality = []
for r in runs:
    m = r['meta']
    start = m['start'] - m['start'] % STEP_SEC
    r['T'] = int(math.ceil((m['end'] - start) / STEP_SEC))
    for key in ('node', 'svc', 'edge'):
        if not r[key].empty:
            r[key] = to_grid(r[key], m)
            r[key] = r[key][(r[key].t >= 0) & (r[key].t < r['T'])]
    loc = r['locust']
    if not loc.empty:
        loc = loc.assign(t=((loc['Timestamp'] - start) // STEP_SEC).astype(int))
        r['locust_grid'] = loc.groupby('t').agg(users=('User Count', 'mean'), locust_rps=('Requests/s', 'mean'),
                                               locust_p95_ms=('95%', 'mean'))
    else:
        r['locust_grid'] = pd.DataFrame()

    prom_cov = r['node'].dropna(subset=['cpu_cores']).groupby(['t', 'service']).ngroups / (r['T'] * len(SERVICES)) \
        if not r['node'].empty else 0.0
    fe = r['svc'][r['svc'].service == 'frontend'] if not r['svc'].empty else pd.DataFrame()
    traced = fe['request_count'].sum() if not fe.empty else 0
    sent = (r['locust_grid']['locust_rps'].sum() * STEP_SEC) if not r['locust_grid'].empty else np.nan
    r['jaeger_coverage'] = traced / sent if sent and sent > 0 else np.nan
    quality.append({'scenario': r['scenario'], 'run': r['run_id'], 'T': r['T'],
                    'prometheus_coverage': round(prom_cov, 3),
                    'jaeger_coverage': round(float(r['jaeger_coverage']), 3),
                    'truncated_windows': r['report'].get('jaeger_truncated_windows', 0),
                    'locust_exit': r['meta'].get('locust_exit_code')})

quality = pd.DataFrame(quality)
bad = quality[(quality.prometheus_coverage < 0.9) | (quality.jaeger_coverage < JAEGER_COVERAGE_MIN)]
if len(bad):
    print('⚠️ Các run có độ phủ thấp, cân nhắc loại bỏ hoặc chạy lại:')
    display(bad)
quality

## 5. Đặc trưng (features) và lý do lựa chọn

Mọi đặc trưng đầu vào đều phải **quan sát được ở pha triển khai thật**, vì controller MAPE-K đọc trực tiếp từ Prometheus/Jaeger. Vì thế số user của Locust **không** được dùng làm đặc trưng: hệ thống thật không biết có bao nhiêu người dùng, và đưa nó vào sẽ làm rò rỉ thông tin của bộ sinh tải. Locust chỉ dùng để kiểm tra độ phủ.

### 5.1 Đặc trưng nút $X_\tau \in \mathbb{R}^{N \times F}$

| Nhóm | Đặc trưng | Nguồn / công thức | Mục tiêu |
|---|---|---|---|
| Tải | `rps_in` | Jaeger: số span `server` / 10s | **Nhu cầu tải trực tiếp** của service; cũng là target chính. Không bị chặn bởi limit nên phản ánh nhu cầu thật tốt hơn CPU |
| Tải | `rps_in_delta` | $rps_t - rps_{t-1}$ | Xu hướng tức thời. Giúp mô hình nhận ra đầu một spike hay burst trước khi GRU tích lũy đủ ngữ cảnh |
| Tải | `rps_per_replica` | `rps_in / replicas` | Tải trên mỗi pod, quyết định bão hòa. Cùng `rps_in` nhưng ít replica thì chịu áp lực lớn hơn |
| Hiệu năng | `latency_p50_ms`, `latency_p95_ms` | Jaeger: phân vị thời lượng span server | Độ trễ đuôi là **tín hiệu SLO-aware**. Độ trễ tăng ở downstream sẽ lan ngược lên upstream (cascading), đây là mối quan hệ GAT cần học |
| Hiệu năng | `error_rate` | lỗi / request (HTTP 5xx, gRPC ≠ 0) | Dấu hiệu quá tải hoặc lỗi dây chuyền. Là thành phần SLO violation trong reward PPO |
| Tài nguyên | `cpu_cores` | cAdvisor `rate(container_cpu_usage_seconds_total)`, chỉ container ứng dụng | **Nhu cầu tài nguyên**, target thứ hai. PPO quy đổi sang số replica cần |
| Tài nguyên | `cpu_util_request` | `cpu_cores / cpu_request_cores` | Đúng đại lượng HPA dùng. Chuẩn hóa giữa các service có request khác nhau, nên so sánh được với baseline HPA |
| Tài nguyên | `cpu_throttle_ratio` | throttled periods / CFS periods | Báo bão hòa ở **limit**. Khi CPU bị chặn, `cpu_cores` đứng yên nhưng latency tăng; feature này giúp phân biệt "tải thấp" với "đã nghẽn" |
| Tài nguyên | `cpu_sidecar_cores` | CPU của container `istio-proxy` | Tỉ lệ thuận với lưu lượng qua Envoy. Là tín hiệu tải dự phòng khi Jaeger rơi span, và là tín hiệu duy nhất cho `redis-cart` |
| Tài nguyên | `mem_mib` | `container_memory_working_set_bytes` | Áp lực bộ nhớ (adservice dùng JVM, các service Python). Cũng cảnh báo trước OOMKill |
| Tài nguyên | `net_rx_kBps`, `net_tx_kBps` | cAdvisor network theo pod | Khối lượng traffic, gồm cả TCP không có span (`cartservice → redis-cart`) |
| Năng lực | `replicas` | KSM `kube_deployment_status_replicas_available` | Năng lực hiện tại, cũng là biến PPO điều khiển. Mô hình cần biết replica để tách ảnh hưởng của scaling khỏi biến động tải |
| Năng lực | `restarts_delta` | chênh lệch `kube_pod_container_status_restarts_total` | Sự kiện bất ổn (crash, OOM) gây gián đoạn năng lực đột ngột |

### 5.2 Đặc trưng cạnh $E^{attr}_\tau \in \mathbb{R}^{M \times F_e}$

| Đặc trưng | Nguồn | Mục tiêu |
|---|---|---|
| `call_rate` | Jaeger: số span `client` A→B / 10s | **Trọng số phụ thuộc động**. Cạnh gọi nhiều thì truyền nhiều tải hơn; GAT dùng làm tín hiệu khi tính attention |
| `call_ratio` | `call_rate / rps_in(A)` | Hệ số fan-out: mỗi request vào A sinh bao nhiêu lời gọi sang B. Đại lượng khá ổn định, giúp mô hình **lan truyền tải dự báo xuống downstream** |
| `edge_error_rate` | lỗi / lời gọi | Lỗi lan truyền dọc chuỗi gọi |
| `edge_latency_p95_ms` | p95 thời lượng span client | Độ trễ mà upstream thấy ở downstream (gồm mạng + mesh), là kênh truyền của hiệu ứng dây chuyền |

### 5.3 Đặc trưng toàn đồ thị $g_\tau$ (cho PPO/SLO)

`e2e_rps`, `e2e_latency_p95_ms`, `e2e_error_rate` được lấy tại `frontend`, cửa vào của người dùng. `slo_violation` = 1 khi p95 > 300ms hoặc lỗi > 1%. `total_replicas` là proxy chi phí. Mô hình dự báo có thể ghép các đặc trưng này vào mọi nút; PPO dùng trực tiếp trong state và reward.

### 5.4 Target $Y$

`rps_in` (nhu cầu tải) và `cpu_cores` (nhu cầu tài nguyên) của từng service tại $t+1, \dots, t+h$. Có hai bản: `Y` (đã biến đổi, dùng để huấn luyện) và `Y_raw` (đơn vị gốc, dùng để tính **MAE/RMSE** như đề cương). Lưu ý `cpu_cores` bị chặn bởi limit khi bão hòa, vì vậy `rps_in` là target bền vững hơn khi tải vượt năng lực.

In [ ]:
# name: (nhóm, phép biến đổi). 'log1p' cho đại lượng không âm, đuôi dài; 'slog1p' cho đại lượng có dấu;
# 'none' cho tỉ lệ đã nằm trong [0, 1] hoặc số đếm nhỏ.
NODE_FEATURES = {
    'rps_in': ('load', 'log1p'), 'rps_in_delta': ('load', 'slog1p'), 'rps_per_replica': ('load', 'log1p'),
    'latency_p50_ms': ('perf', 'log1p'), 'latency_p95_ms': ('perf', 'log1p'), 'error_rate': ('perf', 'none'),
    'cpu_cores': ('resource', 'log1p'), 'cpu_util_request': ('resource', 'log1p'),
    'cpu_throttle_ratio': ('resource', 'none'), 'cpu_sidecar_cores': ('resource', 'log1p'),
    'mem_mib': ('resource', 'log1p'), 'net_rx_kBps': ('resource', 'log1p'), 'net_tx_kBps': ('resource', 'log1p'),
    'replicas': ('capacity', 'none'), 'restarts_delta': ('capacity', 'none'),
}
EDGE_FEATURES = {'call_rate': 'log1p', 'call_ratio': 'log1p', 'edge_error_rate': 'none',
                 'edge_latency_p95_ms': 'log1p'}
GRAPH_FEATURES = {'e2e_rps': 'log1p', 'e2e_latency_p95_ms': 'log1p', 'e2e_error_rate': 'none',
                  'slo_violation': 'none', 'total_replicas': 'none'}
NF, GF = list(NODE_FEATURES), list(GRAPH_FEATURES)
print(len(NF), 'node features |', len(EDGE_FEATURES), 'edge features |', len(GF), 'graph features')

## 6. Xây dựng Service Graph

Tập cạnh = **topology tĩnh** ∪ **cạnh quan sát được** trong trace (bất kỳ run nào, tổng lời gọi > 0).
- Giữ cạnh tĩnh kể cả khi không có span (ví dụ `cartservice → redis-cart` là TCP). GAT vẫn truyền thông tin qua cạnh, chỉ có đặc trưng bằng 0.
- Thêm cạnh quan sát được mà topology chưa có, để đồ thị khớp với hành vi thật của ứng dụng.

Cạnh được lưu theo dạng `edge_index` [2, M] (source → target, kiểu PyTorch Geometric). Lớp GAT có thể dùng chiều ngược (target → source) để mô hình hóa áp lực lan ngược từ downstream lên upstream. Việc đảo chiều này nên làm ở code mô hình, ở đây giữ nguyên chiều gọi.

In [ ]:
observed = set()
for r in runs:
    if not r['edge'].empty:
        g = r['edge'].groupby(['source', 'target'])['call_count'].sum()
        observed |= {e for e, n in g.items() if n > 0 and e[0] in SVC_IDX and e[1] in SVC_IDX and e[0] != e[1]}
EDGES = list(STATIC_EDGES) + sorted(observed - set(STATIC_EDGES))
EDGE_IDX = {e: i for i, e in enumerate(EDGES)}
edge_index = np.array([[SVC_IDX[a] for a, _ in EDGES], [SVC_IDX[b] for _, b in EDGES]], dtype=np.int64)
print(f'{len(EDGES)} cạnh ({len(STATIC_EDGES)} tĩnh, {len(EDGES) - len(STATIC_EDGES)} chỉ thấy trong trace)')
missing_static = [e for e in STATIC_EDGES if e not in observed and e[1] != 'redis-cart']
if missing_static:
    print('⚠️ Cạnh tĩnh không thấy trong trace (Jaeger mất span hoặc luồng không được gọi):', missing_static)

## 7. Tạo tensor đặc trưng cho từng run

Với mỗi run: `X[T, N, F]`, `E[T, M, Fe]`, `G[T, Fg]`, `Yraw[T, N, K]` (giá trị **thô**, chưa biến đổi).

Xử lý thiếu dữ liệu:
- **Gauge** (`mem`, `replicas`, `cpu_request`) được forward-fill tối đa 3 bước (30s), vì giá trị đổi chậm.
- **Rate/đếm** (`rps`, `cpu`, `call_rate`) thiếu nghĩa là không có hoạt động, nên điền 0.
- **Latency** khi không có request thì không xác định, nên điền 0. Mô hình phân biệt được nhờ `rps_in = 0`.

In [ ]:
def run_tensors(r):
    T, N, M = r['T'], len(SERVICES), len(EDGES)
    full = pd.MultiIndex.from_product([range(T), SERVICES], names=['t', 'service'])

    node = r['node'].groupby(['t', 'service']).mean(numeric_only=True).reindex(full) \
        if not r['node'].empty else pd.DataFrame(index=full)
    for c in ['cpu_cores', 'cpu_sidecar_cores', 'cpu_throttle_ratio', 'mem_bytes', 'net_rx_bps', 'net_tx_bps',
              'cpu_request_cores', 'cpu_limit_cores', 'restarts_total', 'replicas', 'replicas_desired']:
        if c not in node:
            node[c] = np.nan
    gauges = ['mem_bytes', 'cpu_request_cores', 'cpu_limit_cores', 'restarts_total', 'replicas', 'replicas_desired']
    node[gauges] = node[gauges].groupby(level='service').ffill(limit=3).groupby(level='service').bfill(limit=3)
    node = node.fillna({'replicas': 1}).fillna(0.0)

    svc = r['svc'].groupby(['t', 'service']).sum(numeric_only=True).reindex(full) \
        if not r['svc'].empty else pd.DataFrame(index=full, columns=['request_count', 'error_count',
                                                                      'latency_p50_ms', 'latency_p95_ms'])
    svc = svc.astype(float)
    # Phân vị không cộng được: lấy lại theo bản ghi gốc (mỗi (t, service) chỉ có 1 dòng từ collector)
    if not r['svc'].empty:
        lat = r['svc'].groupby(['t', 'service'])[['latency_p50_ms', 'latency_p95_ms']].max().reindex(full)
        svc[['latency_p50_ms', 'latency_p95_ms']] = lat
    svc = svc.fillna(0.0)

    cov = 1.0
    if CORRECT_COVERAGE and pd.notna(r['jaeger_coverage']) and r['jaeger_coverage'] > 0:
        cov = r['jaeger_coverage']

    df = pd.DataFrame(index=full)
    df['rps_in'] = svc['request_count'] / STEP_SEC / cov
    df['rps_in_delta'] = df['rps_in'].groupby(level='service').diff().fillna(0.0)
    df['replicas'] = node['replicas'].clip(lower=1)
    df['rps_per_replica'] = df['rps_in'] / df['replicas']
    df['latency_p50_ms'] = svc['latency_p50_ms']
    df['latency_p95_ms'] = svc['latency_p95_ms']
    df['error_rate'] = (svc['error_count'] / svc['request_count'].where(svc['request_count'] > 0)).fillna(0).clip(0, 1)
    df['cpu_cores'] = node['cpu_cores']
    df['cpu_util_request'] = (node['cpu_cores'] / node['cpu_request_cores'].where(node['cpu_request_cores'] > 0)).fillna(0)
    df['cpu_throttle_ratio'] = node['cpu_throttle_ratio'].clip(0, 1)
    df['cpu_sidecar_cores'] = node['cpu_sidecar_cores']
    df['mem_mib'] = node['mem_bytes'] / 2**20
    df['net_rx_kBps'] = node['net_rx_bps'] / 1024
    df['net_tx_kBps'] = node['net_tx_bps'] / 1024
    df['restarts_delta'] = node['restarts_total'].groupby(level='service').diff().fillna(0).clip(lower=0)

    X = df[NF].to_numpy(dtype=np.float32).reshape(T, N, len(NF))
    Yraw = df[TARGETS].to_numpy(dtype=np.float32).reshape(T, N, len(TARGETS))

    E = np.zeros((T, M, len(EDGE_FEATURES)), dtype=np.float32)
    if not r['edge'].empty:
        e = r['edge'][r['edge'].apply(lambda x: (x.source, x.target) in EDGE_IDX, axis=1)]
        for row in e.itertuples(index=False):
            k = EDGE_IDX[(row.source, row.target)]
            calls = row.call_count / STEP_SEC / cov
            src_rps = df.at[(row.t, row.source), 'rps_in']
            E[row.t, k] = [calls, calls / src_rps if src_rps > 0 else 0.0,
                           row.error_count / row.call_count if row.call_count else 0.0,
                           0.0 if pd.isna(row.latency_p95_ms) else row.latency_p95_ms]

    fe = df.xs('frontend', level='service')
    G = pd.DataFrame({
        'e2e_rps': fe['rps_in'], 'e2e_latency_p95_ms': fe['latency_p95_ms'], 'e2e_error_rate': fe['error_rate'],
        'slo_violation': ((fe['latency_p95_ms'] > LATENCY_SLO_MS) | (fe['error_rate'] > ERROR_RATE_SLO)).astype(float),
        'total_replicas': df['replicas'].groupby(level='t').sum(),
    })[GF].to_numpy(dtype=np.float32)

    keep = slice(WARMUP_DROP, T)
    tidy = df.reset_index().assign(scenario=r['scenario'], run=r['run_id'])
    tidy = tidy[tidy.t >= WARMUP_DROP]
    return X[keep], E[keep], G[keep], Yraw[keep], tidy


for r in runs:
    r['X'], r['E'], r['G'], r['Yraw'], r['tidy'] = run_tensors(r)
tidy_all = pd.concat([r['tidy'] for r in runs], ignore_index=True)
print('X', runs[0]['X'].shape, '| E', runs[0]['E'].shape, '| G', runs[0]['G'].shape, '| Yraw', runs[0]['Yraw'].shape)
tidy_all.groupby('service')[['rps_in', 'cpu_cores', 'latency_p95_ms', 'replicas']].mean().round(3)

### Kiểm tra trực quan một run mỗi kịch bản
`rps_in` của frontend (tải vào hệ thống), CPU từng service và số replica theo thời gian. Nếu `replicas` là hằng số trong mọi run, dữ liệu được thu **không có autoscaler**. Mô hình vẫn học được quan hệ tải–tài nguyên, nhưng PPO sẽ thiếu ví dụ về tác động của scaling (xem README, mục khuyến nghị).

In [ ]:
scen = sorted({r['scenario'] for r in runs})
fig, axes = plt.subplots(3, len(scen), figsize=(5.5 * len(scen), 9), sharex='col', squeeze=False)
for j, sc in enumerate(scen):
    d = tidy_all[(tidy_all.scenario == sc) & (tidy_all.run == tidy_all[tidy_all.scenario == sc].run.min())]
    minutes = lambda x: x.t * STEP_SEC / 60
    fe = d[d.service == 'frontend']
    axes[0, j].plot(minutes(fe), fe.rps_in, color='C3')
    axes[0, j].set_title(f'{sc.upper()}: frontend rps_in')
    for s in SERVICES:
        ds = d[d.service == s]
        axes[1, j].plot(minutes(ds), ds.cpu_cores, lw=0.9, label=s)
        axes[2, j].plot(minutes(ds), ds.replicas, lw=0.9)
    axes[1, j].set_title('cpu_cores theo service')
    axes[2, j].set_title('replicas')
    axes[2, j].set_xlabel('phút')
axes[1, -1].legend(fontsize=6, ncol=2, loc='upper right')
fig.tight_layout()
plt.show()

In [ ]:
# Service Graph: độ dày cạnh = call_rate trung bình, kích thước nút = rps_in trung bình
mean_rate = np.concatenate([r['E'][..., 0] for r in runs]).mean(axis=0)
mean_rps = tidy_all.groupby('service').rps_in.mean().reindex(SERVICES).fillna(0).to_numpy()
ang = np.linspace(0, 2 * np.pi, len(SERVICES) - 1, endpoint=False)
pos = {'frontend': (0.0, 0.0)}
for a, s in zip(ang, [s for s in SERVICES if s != 'frontend']):
    pos[s] = (np.cos(a), np.sin(a))
fig, ax = plt.subplots(figsize=(8, 8))
w = mean_rate / (mean_rate.max() or 1)
for (a, b), wi in zip(EDGES, w):
    (x0, y0), (x1, y1) = pos[a], pos[b]
    ax.annotate('', xy=(x1, y1), xytext=(x0, y0),
                arrowprops=dict(arrowstyle='-|>', lw=0.6 + 5 * wi, color='C0', alpha=0.35 + 0.6 * wi,
                                shrinkA=14, shrinkB=14))
size = 300 + 2500 * mean_rps / (mean_rps.max() or 1)
ax.scatter([pos[s][0] for s in SERVICES], [pos[s][1] for s in SERVICES], s=size, color='C1', zorder=3, alpha=0.85)
for s in SERVICES:
    ax.text(pos[s][0], pos[s][1] - 0.13, s, ha='center', fontsize=8)
ax.set_title('Service Graph (độ dày cạnh ∝ call_rate, kích thước nút ∝ rps_in)')
ax.axis('off')
plt.show()

## 8. Chia tập train / val / test

- **`by_run` (mặc định):** trong mỗi kịch bản, các run được chia 70/15/15 (mỗi tập ít nhất 1 run khi có ≥ 3 run). Cả 3 tập đều có đủ các kịch bản. Nếu một kịch bản có < 3 run, chia **theo thời gian** bên trong run (đầu run cho train, cuối run cho test) và bỏ các cửa sổ nằm vắt qua ranh giới.
- **`by_scenario`:** toàn bộ `TEST_SCENARIOS` làm test, phần còn lại chia train/val. Dùng để đánh giá khả năng tổng quát hóa với mẫu tải chưa thấy, ví dụ huấn luyện trên normal + bursty rồi kiểm tra trên spike.

Bộ chuẩn hóa chỉ được fit trên **train**, tránh rò rỉ thống kê từ val/test.

In [ ]:
rng = np.random.default_rng(SEED)
for r in runs:
    r['split_t'] = None
by_scen = {}
for r in runs:
    by_scen.setdefault(r['scenario'], []).append(r)

for sc, rs in by_scen.items():
    rs = sorted(rs, key=lambda x: x['run_id'])
    if SPLIT_MODE == 'by_scenario':
        if sc in TEST_SCENARIOS:
            labels = ['test'] * len(rs)
        else:
            n_val = max(1, round(len(rs) * SPLIT_FRAC['val'])) if len(rs) >= 2 else 0
            labels = ['train'] * (len(rs) - n_val) + ['val'] * n_val
        for r, lab in zip(rs, labels):
            r['split'] = lab
    elif len(rs) >= 3:
        idx = rng.permutation(len(rs))
        n_test = max(1, round(len(rs) * SPLIT_FRAC['test']))
        n_val = max(1, round(len(rs) * SPLIT_FRAC['val']))
        lab = np.array(['train'] * len(rs), dtype=object)
        lab[idx[:n_test]] = 'test'
        lab[idx[n_test:n_test + n_val]] = 'val'
        for r, l in zip(rs, lab):
            r['split'] = l
    else:  # chia theo thời gian trong run
        for r in rs:
            T = len(r['X'])
            a, b = int(T * SPLIT_FRAC['train']), int(T * (SPLIT_FRAC['train'] + SPLIT_FRAC['val']))
            r['split'] = 'time'
            r['split_t'] = np.array(['train'] * a + ['val'] * (b - a) + ['test'] * (T - b), dtype=object)

pd.DataFrame([{'scenario': r['scenario'], 'run': r['run_id'], 'split': r['split']} for r in runs]) \
    .pivot_table(index='scenario', columns='split', values='run', aggfunc='count', fill_value=0)

## 9. Làm sạch và chuẩn hóa

1. **Clip ngoại lai**: chặn trên tại phân vị 99.9 của train cho từng đặc trưng. Một vài giá trị latency cực lớn (timeout) không được phép chi phối thang đo.
2. **Biến đổi**: `log1p` cho đại lượng không âm có đuôi dài (rps, latency, bytes, cpu); `sign·log1p|x|` cho `rps_in_delta`; tỉ lệ thì giữ nguyên.
3. **Z-score** theo từng đặc trưng (gộp mọi service), fit trên train. Dùng chung thang cho mọi nút để giữ khác biệt tương đối giữa các service. Đây là thông tin GAT cần, ví dụ productcatalog luôn tải cao hơn email.

Target dùng cùng phép biến đổi. Tham số được lưu vào `metadata.json` để đảo ngược khi tính MAE/RMSE và khi controller chạy online.

In [ ]:
def _transform(x, kind):
    if kind == 'log1p':
        return np.log1p(np.clip(x, 0, None))
    if kind == 'slog1p':
        return np.sign(x) * np.log1p(np.abs(x))
    return x


def train_mask(r, T):
    if r['split'] == 'time':
        return r['split_t'] == 'train'
    return np.full(T, r['split'] == 'train')


def fit_scaler(arrays, kinds):
    # arrays: list of [T, ..., F]; returns clip, mean, std per feature
    flat = np.concatenate([a.reshape(-1, a.shape[-1]) for a in arrays])
    clip = np.nanpercentile(flat, 99.9, axis=0)
    tr = np.stack([_transform(np.minimum(flat[:, i], clip[i]), k) for i, k in enumerate(kinds)], axis=1)
    mean, std = tr.mean(axis=0), tr.std(axis=0)
    std[std < 1e-6] = 1.0
    return {'clip': clip, 'mean': mean, 'std': std, 'kinds': list(kinds)}


def apply_scaler(a, sc):
    out = np.empty_like(a, dtype=np.float32)
    for i, k in enumerate(sc['kinds']):
        out[..., i] = (_transform(np.minimum(a[..., i], sc['clip'][i]), k) - sc['mean'][i]) / sc['std'][i]
    return out


tr_X = [r['X'][train_mask(r, len(r['X']))] for r in runs]
tr_E = [r['E'][train_mask(r, len(r['E']))] for r in runs]
tr_G = [r['G'][train_mask(r, len(r['G']))] for r in runs]
tr_Y = [r['Yraw'][train_mask(r, len(r['Yraw']))] for r in runs]
SC = {
    'node': fit_scaler(tr_X, [NODE_FEATURES[f][1] for f in NF]),
    'edge': fit_scaler(tr_E, list(EDGE_FEATURES.values())),
    'graph': fit_scaler(tr_G, list(GRAPH_FEATURES.values())),
    'target': fit_scaler(tr_Y, [NODE_FEATURES[f][1] for f in TARGETS]),
}
for r in runs:
    r['Xs'], r['Es'] = apply_scaler(r['X'], SC['node']), apply_scaler(r['E'], SC['edge'])
    r['Gs'], r['Ys'] = apply_scaler(r['G'], SC['graph']), apply_scaler(r['Yraw'], SC['target'])

pd.DataFrame({'transform': SC['node']['kinds'], 'clip_p99.9': SC['node']['clip'], 'mean': SC['node']['mean'],
              'std': SC['node']['std']}, index=NF).round(4)

In [ ]:
# Phân phối trước/sau biến đổi của vài đặc trưng tiêu biểu (train)
show = ['rps_in', 'latency_p95_ms', 'cpu_cores', 'net_rx_kBps']
raw = np.concatenate([x.reshape(-1, len(NF)) for x in tr_X])
scaled = apply_scaler(raw, SC['node'])
fig, axes = plt.subplots(2, len(show), figsize=(16, 5.5))
for j, f in enumerate(show):
    i = NF.index(f)
    axes[0, j].hist(raw[:, i], bins=60, color='C0')
    axes[0, j].set_title(f'{f} (thô)')
    axes[1, j].hist(scaled[:, i], bins=60, color='C2')
    axes[1, j].set_title(f'{f} (sau {SC["node"]["kinds"][i]} + z-score)')
fig.tight_layout()
plt.show()

# Tương quan giữa các đặc trưng nút: cặp |r| > 0.95 là ứng viên loại bỏ (dư thừa) trong ablation
corr = pd.DataFrame(scaled, columns=NF).corr()
fig, ax = plt.subplots(figsize=(9, 7.5))
im = ax.imshow(corr, cmap='RdBu_r', vmin=-1, vmax=1)
ax.set_xticks(range(len(NF)), NF, rotation=75, fontsize=8)
ax.set_yticks(range(len(NF)), NF, fontsize=8)
fig.colorbar(im)
ax.set_title('Tương quan Pearson giữa các đặc trưng nút (train, đã chuẩn hóa)')
plt.show()
high = [(a, b, round(corr.loc[a, b], 3)) for i, a in enumerate(NF) for b in NF[i + 1:] if abs(corr.loc[a, b]) > 0.95]
print('Cặp tương quan rất cao:', high or 'không có')

## 10. Cắt cửa sổ trượt

Từ mỗi run, mỗi mẫu gồm:
- Đầu vào: `X[s : s+w]`, `E[s : s+w]`, `G[s : s+w]`.
- Target: `Y[s+w : s+w+h]`.

Stride bằng 1. Cửa sổ **không vượt qua ranh giới run**, và với chia theo thời gian thì cũng không vượt qua ranh giới train/val/test.

| Tensor | Shape | Ý nghĩa |
|---|---|---|
| `X` | [S, w, N, F] | chuỗi đặc trưng nút |
| `E` | [S, w, M, Fe] | chuỗi đặc trưng cạnh |
| `G` | [S, w, Fg] | chuỗi đặc trưng toàn đồ thị |
| `Y` | [S, h, N, K] | target đã chuẩn hóa (để huấn luyện) |
| `Y_raw` | [S, h, N, K] | target đơn vị gốc (để tính MAE/RMSE) |
| `edge_index` | [2, M] | danh sách cạnh source → target |

In [ ]:
def windows(r, split):
    T = len(r['Xs'])
    lab = r['split_t'] if r['split'] == 'time' else np.full(T, r['split'], dtype=object)
    idx = [s for s in range(T - WINDOW - HORIZON + 1) if (lab[s:s + WINDOW + HORIZON] == split).all()]
    if not idx:
        return None
    take = lambda a, off, n: np.stack([a[s + off:s + off + n] for s in idx])
    return {'X': take(r['Xs'], 0, WINDOW), 'E': take(r['Es'], 0, WINDOW), 'G': take(r['Gs'], 0, WINDOW),
            'Y': take(r['Ys'], WINDOW, HORIZON), 'Y_raw': take(r['Yraw'], WINDOW, HORIZON),
            'info': np.array([(r['scenario'], r['run_id'], s) for s in idx], dtype=object)}


data = {}
for split in ['train', 'val', 'test']:
    parts = [w for w in (windows(r, split) for r in runs) if w is not None]
    if not parts:
        print(f'⚠️ {split}: không có cửa sổ nào (cần thêm run)')
        continue
    data[split] = {k: np.concatenate([p[k] for p in parts]) for k in parts[0]}

pd.DataFrame({s: {k: v.shape for k, v in d.items()} for s, d in data.items()})

In [ ]:
# Ví dụ một mẫu: 2 phút quá khứ (xanh) và 60s cần dự báo (đỏ) của rps_in và cpu_cores tại frontend
d = data['train']
k = int(np.argmax(d['Y_raw'][:, :, SVC_IDX['frontend'], 0].std(axis=1)))  # mẫu biến động mạnh nhất
fig, axes = plt.subplots(1, 2, figsize=(13, 3.5))
for j, (name, xi) in enumerate([('rps_in', 0), ('cpu_cores', NF.index('cpu_cores'))]):
    past = d['X'][k, :, SVC_IDX['frontend'], xi] * SC['node']['std'][xi] + SC['node']['mean'][xi]
    axes[j].plot(range(-WINDOW + 1, 1), np.expm1(past), 'o-', label='quá khứ (w)')
    axes[j].plot(range(1, HORIZON + 1), d['Y_raw'][k, :, SVC_IDX['frontend'], j], 's-', color='C3', label='target (h)')
    axes[j].axvline(0.5, color='gray', ls='--')
    axes[j].set_title(f'frontend {name} | mẫu {tuple(d["info"][k])}')
    axes[j].set_xlabel('bước (10s)')
    axes[j].legend()
fig.tight_layout()
plt.show()

## 11. Baseline tham chiếu

Trước khi huấn luyện GAT-GRU, cần biết hai mốc MAE/RMSE:
- **Persistence**: $\hat y_{t+k} = y_t$. Mô hình phải tốt hơn mốc này mới có ích cho scaling chủ động.
- **Trung bình cửa sổ**: $\hat y_{t+k} = \frac{1}{w}\sum y_{t-w+1:t}$.

Các số này phục vụ phần đánh giá dự báo và ablation trong đề cương.

In [ ]:
def baseline_metrics(d):
    rows = []
    for j, tgt in enumerate(TARGETS):
        xi = NF.index(tgt)
        last = np.expm1(d['X'][:, -1:, :, xi] * SC['node']['std'][xi] + SC['node']['mean'][xi])
        mean_w = np.expm1(d['X'][:, :, :, xi] * SC['node']['std'][xi] + SC['node']['mean'][xi]).mean(1, keepdims=True)
        y = d['Y_raw'][..., j]
        for name, pred in [('persistence', last), ('window_mean', mean_w)]:
            err = np.broadcast_to(pred, y.shape) - y
            rows.append({'target': tgt, 'baseline': name, 'MAE': np.abs(err).mean(),
                         'RMSE': np.sqrt((err ** 2).mean()),
                         'MAE@h=1': np.abs(err[:, 0]).mean(), f'MAE@h={HORIZON}': np.abs(err[:, -1]).mean()})
    return pd.DataFrame(rows)


# Lưu ý: target được clip ở p99.9 trước khi biến đổi, nên expm1(X) có thể thấp hơn giá trị thô ở các đỉnh cực trị.
baseline_metrics(data['test'] if 'test' in data else data['val']).round(4)

## 12. Lưu dataset

- `processed/gatgru_dataset.npz`: các tensor `X_*`, `E_*`, `G_*`, `Y_*`, `Y_raw_*`, `info_*` cho từng split, cùng `edge_index`.
- `processed/metadata.json`: tên service, cạnh, tên đặc trưng, phép biến đổi, tham số scaler, `WINDOW`/`HORIZON`, run thuộc split nào.
- `processed/timeseries_raw.csv`: bảng đã căn chỉnh ở đơn vị gốc (scenario, run, t, service, đặc trưng). Dùng để xây **môi trường PPO offline** (transition giữa tải, replica và SLO).

Nạp lại ở bước huấn luyện:
```python
d = np.load('processed/gatgru_dataset.npz', allow_pickle=True)
X, E, Y, edge_index = d['X_train'], d['E_train'], d['Y_train'], d['edge_index']
# X: [S, w, N, F] -> mỗi bước: GAT trên (X[:, τ], edge_index, E[:, τ]) -> GRU theo τ -> đầu ra [S, h, N, K]
```

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
arrays = {'edge_index': edge_index}
for split, d in data.items():
    for k, v in d.items():
        arrays[f'{k}_{split}'] = v
np.savez_compressed(OUTPUT_DIR / 'gatgru_dataset.npz', **arrays)


def _js(sc):
    return {k: (v.tolist() if isinstance(v, np.ndarray) else v) for k, v in sc.items()}


metadata = {
    'services': SERVICES, 'edges': EDGES, 'static_edges': STATIC_EDGES,
    'node_features': {f: {'group': g, 'transform': t} for f, (g, t) in NODE_FEATURES.items()},
    'edge_features': EDGE_FEATURES, 'graph_features': GRAPH_FEATURES, 'targets': TARGETS,
    'step_sec': STEP_SEC, 'window': WINDOW, 'horizon': HORIZON, 'warmup_drop': WARMUP_DROP,
    'slo': {'latency_p95_ms': LATENCY_SLO_MS, 'error_rate': ERROR_RATE_SLO},
    'split_mode': SPLIT_MODE, 'splits': {f"{r['scenario']}/{r['run_id']}": r['split'] for r in runs},
    'scalers': {k: _js(v) for k, v in SC.items()}, 'demo': DEMO,
    'jaeger_coverage': {f"{r['scenario']}/{r['run_id']}": r['jaeger_coverage'] for r in runs},
}
(OUTPUT_DIR / 'metadata.json').write_text(json.dumps(metadata, indent=2, default=float))
tidy_all.to_csv(OUTPUT_DIR / 'timeseries_raw.csv', index=False)
print('Đã lưu vào', OUTPUT_DIR.resolve())
for p in sorted(OUTPUT_DIR.iterdir()):
    print(f'  {p.name:<24} {p.stat().st_size / 1e6:8.2f} MB')

## 13. Gợi ý ablation study

Các nhóm đặc trưng trong `metadata.json → node_features[*].group` cho phép bỏ từng nhóm để đo đóng góp:

| Thí nghiệm | Cách làm | Câu hỏi trả lời |
|---|---|---|
| Không đồ thị | Thay GAT bằng MLP trên từng nút (GRU thuần) | Quan hệ phụ thuộc có giúp dự báo tốt hơn không? |
| Không đặc trưng cạnh | Đặt `E = 0`, chỉ dùng `edge_index` | Trace Jaeger (`call_rate`, latency cạnh) có giá trị gì? |
| Chỉ topology tĩnh | Bỏ các cạnh chỉ thấy trong trace | Đồ thị quan sát động so với đồ thị thiết kế |
| Bỏ nhóm `perf` | Bỏ latency/error | Tín hiệu SLO có giúp dự báo tải không? |
| Bỏ nhóm `resource` | Chỉ dùng tải + replica | CPU/throttle/network bổ sung được gì? |
| Thay đổi `HORIZON` | 1, 3, 6, 12 bước | Độ chính xác giảm thế nào khi dự báo xa hơn? |
| `SPLIT_MODE = 'by_scenario'` | Test trên spike | Mô hình có tổng quát hóa sang mẫu tải chưa thấy không? |